# Bukti minggu pertama — median harga laptop

Satu kategori, satu tahun anggaran. Tidak ada frontend, tidak ada API, tidak ada database.

**Kriteria lulus:**

1. Median laptop masuk akal saat dibaca mata — bukan angka yang jelas salah.
2. Share kategori `lain_lain` cukup kecil untuk kategori yang dituju.
3. Baris yang dibuang punya alasan yang bisa dijelaskan, dan jumlahnya wajar.
4. Outlier teratas benar-benar janggal saat dibaca manual, bukan artefak normalisasi.

Kalau salah satu gagal, berhenti di sini dan perbaiki aturan normalisasi.
Tidak ada stack yang bisa menyelamatkan benchmark yang salah.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

from pipeline import benchmark as bench
from pipeline import normalize
from pipeline.sources import LocalCsvSource
from pipeline.sources.base import to_frame

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(60)

FISCAL_YEAR = 2023
CATEGORY = "laptop"

# Ganti ke folder CSV asli yang kamu unduh. Kalau belum ada, notebook jatuh ke
# fixture kecil di tests/ supaya tetap bisa dijalankan dari awal.
PACKAGE_ROOT = Path.cwd().parent
SOURCE_DIR = Path(os.environ.get("PELINTIR_SOURCE_DIR", PACKAGE_ROOT / "tests" / "fixtures"))

print(f"sumber: {SOURCE_DIR}")

## 1. Ambil data mentah

Tidak ada transformasi di sini. Apa adanya dari sumber.

In [ ]:
source = LocalCsvSource(SOURCE_DIR)
raw_items = to_frame(source.fetch(FISCAL_YEAR))

print(f"{len(raw_items)} baris mentah")
raw_items.head(5)

## 2. Normalisasi

Baris tidak dihapus, hanya diberi label. Yang dibuang harus bisa dijelaskan.

In [ ]:
normalized = normalize.normalize(raw_items)
usable, rejected = normalize.split_usable(normalized)

print(f"terpakai: {len(usable)}   dibuang: {len(rejected)}")
rejected["reject_reason"].value_counts(sort=True)

### Cakupan kategori

Share `lain_lain` adalah ukuran jujur seberapa jauh aturan normalisasi menjangkau.
Angka tinggi bukan bencana — asal kategori yang kamu benchmark bukan bagian dari sisanya.

In [ ]:
coverage = (
    usable.group_by("canonical_category")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum() * 100).round(1).alias("persen"))
    .sort("n", descending=True)
)
coverage

In [ ]:
# Baca 20 baris lain_lain. Kalau ada laptop nyasar di sini, aturannya kurang.
usable.filter(pl.col("canonical_category") == "lain_lain").select(
    "item_description", "canonical_unit", "unit_price"
).head(20)

## 3. Benchmark

SQL-nya ada di `sql/benchmark.sql`, di-versioning git. Grup dengan `n` di bawah
ambang tidak diterbitkan — median dari tiga baris itu anekdot, bukan benchmark.

In [ ]:
benchmarks = bench.run(usable, min_group_size=5)
benchmarks.select(
    "canonical_category",
    "canonical_unit",
    "n",
    "min_price",
    "p25",
    "median_price",
    "p75",
    "max_price",
)

In [ ]:
target = benchmarks.filter(pl.col("canonical_category") == CATEGORY)

if len(target) == 0:
    raise SystemExit(f"Kategori {CATEGORY!r} tidak lolos ambang n. Perluas data atau aturan.")

row = target.row(0, named=True)
print(f"n           : {row['n']}")
print(f"median      : Rp {row['median_price']:,.0f}")
print(f"p25 - p75   : Rp {row['p25']:,.0f} - Rp {row['p75']:,.0f}")
print(f"min - max   : Rp {row['min_price']:,.0f} - Rp {row['max_price']:,.0f}")
print()
print("Pertanyaannya sederhana: kamu percaya median ini?")

## 4. Distribusi

Skala log: harga pengadaan condong berat ke kanan, dan skala linear membuat
seluruh distribusi menempel di satu batang.

In [ ]:
prices = usable.filter(pl.col("canonical_category") == CATEGORY)["unit_price"].to_list()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(prices, bins=30, color="#4c78a8", edgecolor="white")
ax.axvline(row["median_price"], color="#e45756", linewidth=2, label="median")
ax.axvline(row["p25"], color="#9d9d9d", linestyle="--", linewidth=1, label="p25 / p75")
ax.axvline(row["p75"], color="#9d9d9d", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xlabel("harga satuan (rupiah, skala log)")
ax.set_ylabel("jumlah item")
ax.set_title(f"Distribusi harga satuan — {CATEGORY} {FISCAL_YEAR}")
ax.legend()
plt.tight_layout()

## 5. Outlier

Pakai MAD, bukan standar deviasi: satu baris 100x akan menggelembungkan stddev
sampai outlier lain ikut tersembunyi di belakangnya.

**Baca 20 baris ini satu per satu.** Kalau yang muncul cuma kesalahan satuan atau
kategori nyasar, itu bug normalisasi, bukan temuan.

In [ ]:
flagged = bench.outliers(usable, benchmarks, threshold=3.0)

flagged.select(
    "item_description",
    "canonical_category",
    "canonical_unit",
    "quantity",
    "unit_price",
    "median_price",
    pl.col("price_ratio").round(1).alias("x_median"),
    "agency_name",
    "vendor_name",
).head(20)

## Putusan

Isi ini sebelum menulis satu baris pun frontend:

- Median masuk akal? 
- Share `lain_lain` untuk kategori target? 
- Baris terbuang wajar? 
- Outlier teratas janggal secara nyata, bukan bug? 

Kalau keempatnya ya — lanjut ke tiga halaman: detail paket, profil instansi, profil vendor.
Kalau tidak — perbaiki aturan di `src/pipeline/normalize.py`, tambah test-nya, ulangi.